# XAI benchmark v2 su Colab

Ordine consigliato, per non pagare GPU inutilmente:

1. **Runtime solo CPU** (gratuito): passi 1 e 2, i test su modelli finti. Se falliscono, ci si ferma qui.
2. **Runtime GPU**: passo 1, passo 3 (stima dei tempi), poi passo 4 (run senza Attention Flow).
3. **Runtime CPU**: passo 1 e passo 5 (Attention Flow, che usa solo la CPU), infine passo 6.

I risultati vanno su Google Drive: se Colab si disconnette, rilanciando la stessa cella il run riprende da dove era arrivato.

## 1. Codice e dipendenze

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/nicolobachiorri/XAI_Benchmark_for_Sentiment_Analysis.git /content/xai
%cd /content/xai
!git log --oneline -1

In [ ]:
!pip install -q lime pytest
import torch, transformers; print(torch.__version__, transformers.__version__, 'GPU' if torch.cuda.is_available() else 'CPU')

## 2. Test (CPU, pochi minuti, niente download)

In [ ]:
!python -m pytest -q tests

## 3. Stima dei tempi sull'hardware in uso (GPU)

Scarica i cinque modelli e cronometra ogni metodo su tre recensioni di lunghezza tipica.

In [ ]:
!python report.py --estimate --explainers lime shap grad_input attention_rollout lrp

## 4. Run completo senza Attention Flow (GPU)

In [ ]:
OUT = '/content/drive/MyDrive/xai_v2'
!python report.py --explainers lime shap grad_input attention_rollout lrp --out-dir {OUT}

## 5. Attention Flow (runtime CPU)

Il flusso massimo gira su CPU anche con la GPU: conviene un runtime CPU, con un processo per core.

In [ ]:
import os
OUT = '/content/drive/MyDrive/xai_v2'
!python report.py --explainers attention_flow --device cpu --flow-workers {os.cpu_count()} --out-dir {OUT}

## 6. Tabelle finali

Dopo i passi 4 e 5 nella stessa cartella, questa cella ricostruisce le tre tabelle con tutti i metodi.

In [ ]:
import report
from pathlib import Path
tables = report.build_tables(Path('/content/drive/MyDrive/xai_v2'), report.MODEL_ORDER, report.EXPLAINERS)
for name, df in tables.items():
    print(name); display(df)